# Stress Prediction — Fresh Start (v34와 독립적인 새 접근)

v34의 트릭(`max_features=1` 극단 랜덤화, feature copy, 개별 트리 quantile 집계, argmin 1-NN)은
전부 배제하고, 표준적인 방식으로 처음부터 설계했습니다.

**구성**:
1. 최소한의 생리학적 피처(BMI, 맥압, 평균동맥압, 콜레스테롤/혈당비) — 검증된 것만
2. 범주형은 LightGBM 네이티브 category dtype 사용 (원-핫 안 함 -> 컬럼 폭발 없음)
3. 모델 A: LightGBM (`objective="mae"`, early stopping)
4. 모델 B: 표준 RandomForest (`max_features="sqrt"`, `min_samples_leaf=5` — 극단 랜덤화 안 함)
5. 블렌드 가중치: RidgeCV(MSE 최적화라 이전에 실패) 대신 **MAE를 직접 최소화하는 그리드서치**

⚠️ 데이터 파일 경로(`DATA_DIR`)는 실제 환경에 맞게 수정해서 실행해주세요.
저는 실제 데이터에 접근할 수 없어 직접 실행/검증하지 못했습니다.


In [2]:

from __future__ import annotations

import os
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold

TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")     # 실제 경로로 수정
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/fresh")  # 새 폴더
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 1. 데이터 로드 + 최소한의 피처 엔지니어링

BMI/맥압/평균동맥압/콜레스테롤-혈당비는 임상적으로 근거가 뚜렷한 지표라 유지하고,
그 이상은 추가하지 않습니다 (오늘 실험에서 피처를 늘릴수록 안 좋아지는 패턴을 반복 확인했음).


In [3]:

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")


def add_core_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (result["glucose"] + 1e-6)
    result["missing_count"] = result.isna().sum(axis=1)
    return result


raw_X = add_core_features(train.drop(columns=[TARGET, ID_COLUMN])).reset_index(drop=True)
raw_X_test = add_core_features(test.drop(columns=[ID_COLUMN])).reset_index(drop=True)
y = train[TARGET].reset_index(drop=True)

categorical_cols = raw_X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
numeric_cols = raw_X.select_dtypes(include=["number"]).columns.tolist()
print("범주형:", categorical_cols)
print("수치형:", numeric_cols)


범주형: ['gender', 'activity', 'smoke_status', 'medical_history', 'family_medical_history', 'sleep_pattern', 'edu_level']
수치형: ['age', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure', 'diastolic_blood_pressure', 'glucose', 'bone_density', 'mean_working', 'bmi', 'pulse_pressure', 'mean_arterial_pressure', 'cholesterol_glucose_ratio', 'missing_count']


## 2. 모델 A — LightGBM (objective=mae, early stopping)

`!pip install lightgbm`이 필요하면 이 셀 위에서 먼저 실행해주세요.


In [4]:

import lightgbm as lgb


def prep_for_lgbm(train_df, other_df):
    train_local = train_df.copy()
    other_local = other_df.copy()
    for col in categorical_cols:
        train_local[col] = train_local[col].astype("category")
        other_local[col] = pd.Categorical(other_local[col], categories=train_local[col].cat.categories)
    return train_local, other_local


def lgbm_predict_fold(train_df, val_df, y_train, y_val):
    """outer CV의 val fold를 그대로 early stopping 검증셋으로 사용 (Kaggle 표준 관행)."""
    train_local, val_local = prep_for_lgbm(train_df, val_df)
    model = lgb.LGBMRegressor(
        objective="mae",
        n_estimators=3000,
        learning_rate=0.03,
        num_leaves=31,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.05,
        reg_lambda=0.05,
        random_state=RANDOM_SEED,
        verbosity=-1,
    )
    model.fit(
        train_local, y_train,
        eval_set=[(val_local, y_val)],
        eval_metric="mae",
        categorical_feature=categorical_cols if categorical_cols else "auto",
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)],
    )
    return model.predict(val_local, num_iteration=model.best_iteration_), model.best_iteration_


## 3. 모델 B — 표준 RandomForest (극단 랜덤화 없음)

In [5]:

def rf_predict_fold(train_df, val_df, y_train):
    train_local = pd.get_dummies(train_df, columns=categorical_cols, dummy_na=True)
    val_local = pd.get_dummies(val_df, columns=categorical_cols, dummy_na=True)
    val_local = val_local.reindex(columns=train_local.columns, fill_value=0)

    model = RandomForestRegressor(
        n_estimators=800,
        max_features="sqrt",   # v34와 달리 정상적인 랜덤화
        min_samples_leaf=5,    # 극단 과적합 방지
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    model.fit(train_local, y_train)
    return model.predict(val_local)


## 4. 5-Fold CV — 두 모델 OOF 생성 + MAE 직접 최소화하는 블렌드 가중치 탐색

In [6]:

cv = KFold(5, shuffle=True, random_state=RANDOM_SEED)
oof_lgbm = np.zeros(len(y))
oof_rf = np.zeros(len(y))
fold_rows = []
lgbm_best_iterations = []

for fold, (train_idx, val_idx) in enumerate(cv.split(raw_X), start=1):
    train_df = raw_X.iloc[train_idx].reset_index(drop=True)
    val_df = raw_X.iloc[val_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)
    y_val = y.iloc[val_idx].reset_index(drop=True)

    pred_lgbm, best_iter = lgbm_predict_fold(train_df, val_df, y_train, y_val)
    pred_rf = rf_predict_fold(train_df, val_df, y_train)

    oof_lgbm[val_idx] = pred_lgbm
    oof_rf[val_idx] = pred_rf
    lgbm_best_iterations.append(best_iter)

    fold_rows.append({
        "Fold": fold,
        "LGBM_MAE": mean_absolute_error(y_val, pred_lgbm),
        "RF_MAE": mean_absolute_error(y_val, pred_rf),
    })
    print(f"Fold {fold} done")

fold_df = pd.DataFrame(fold_rows)
display(fold_df)
print("개별 모델 평균 MAE:")
print(fold_df[["LGBM_MAE", "RF_MAE"]].mean())

# MAE를 직접 최소화하는 블렌드 가중치 그리드서치 (RidgeCV의 MSE 최적화 문제 회피)
best_w, best_mae = None, np.inf
for w in np.arange(0.0, 1.01, 0.05):
    blended = np.clip(np.round(w * oof_lgbm + (1 - w) * oof_rf, 2), 0.0, 1.0)
    mae = mean_absolute_error(y, blended)
    if mae < best_mae:
        best_mae, best_w = mae, w

print()
print(f"최적 블렌드 가중치 (LGBM 비중): {best_w:.2f}")
print(f"Fresh 접근 CV MAE: {best_mae:.6f}")
print(f"(참고) v34 CV MAE: 0.148033 / v34 Public MAE: 0.127187")


/opt/anaconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 1 done


/opt/anaconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 2 done


/opt/anaconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


...

Fold 4 done


/opt/anaconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 5 done


,Fold,LGBM_MAE,RF_MAE
0,1,0.185681,0.214105
1,2,0.193959,0.223400
2,3,0.200449,0.225162
3,4,0.206596,0.231045
4,5,0.196586,0.224635


개별 모델 평균 MAE:
LGBM_MAE    0.196654
RF_MAE      0.223669
dtype: float64

최적 블렌드 가중치 (LGBM 비중): 1.00
Fresh 접근 CV MAE: 0.196617
(참고) v34 CV MAE: 0.148033 / v34 Public MAE: 0.127187


## 5. 최종 제출 파일 생성

CV MAE가 v34(0.148)보다 낮게 나오면 실행하세요.


In [7]:

final_train_local, final_test_local = prep_for_lgbm(raw_X, raw_X_test)
final_n_estimators = max(100, int(np.mean(lgbm_best_iterations)))
print("최종 LightGBM n_estimators (fold별 best_iteration 평균):", final_n_estimators)

final_lgbm_model = lgb.LGBMRegressor(
    objective="mae", n_estimators=final_n_estimators, learning_rate=0.03,
    num_leaves=31, min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.05, reg_lambda=0.05, random_state=RANDOM_SEED, verbosity=-1,
)
final_lgbm_model.fit(
    final_train_local, y,
    categorical_feature=categorical_cols if categorical_cols else "auto",
)
final_pred_lgbm = final_lgbm_model.predict(final_test_local)

final_train_dummy = pd.get_dummies(raw_X, columns=categorical_cols, dummy_na=True)
final_test_dummy = pd.get_dummies(raw_X_test, columns=categorical_cols, dummy_na=True)
final_test_dummy = final_test_dummy.reindex(columns=final_train_dummy.columns, fill_value=0)
final_rf_model = RandomForestRegressor(
    n_estimators=800, max_features="sqrt", min_samples_leaf=5,
    random_state=RANDOM_SEED, n_jobs=-1,
)
final_rf_model.fit(final_train_dummy, y)
final_pred_rf = final_rf_model.predict(final_test_dummy)

final_prediction = np.clip(np.round(best_w * final_pred_lgbm + (1 - best_w) * final_pred_rf, 2), 0.0, 1.0)

output = submission.copy()
output[TARGET] = final_prediction
submit_path = OUTPUT_DIR / "submit_fresh_lgbm_rf_0806.csv"
output.to_csv(submit_path, index=False, encoding="utf-8")

print("Saved:", submit_path)
print("Prediction range:", final_prediction.min(), final_prediction.max())
display(output.head(10))


최종 LightGBM n_estimators (fold별 best_iteration 평균): 2999
Saved: /Users/bitsaem/Desktop/stress_project/fresh/submit_fresh_lgbm_rf_0806.csv
Prediction range: 0.04 0.99


,ID,stress_score
0,TEST_0000,0.44
1,TEST_0001,0.79
2,TEST_0002,0.16
3,TEST_0003,0.45
4,TEST_0004,0.57
5,TEST_0005,0.46
6,TEST_0006,0.74
7,TEST_0007,0.48
8,TEST_0008,0.52
9,TEST_0009,0.78
